# Amaliy mashgʻulot 18. Sifat metrikalari: BLEU, ROUGE va F1 ni amaliy hisoblash

18-laboratoriyada BLEU, ROUGE va F1 ni tayyor misollarda qadamma-qadam hisobladingiz. Bu yerda barcha metrikalarni noldan yozamiz va kutubxonalar bilan solishtiramiz:

1. BLEU: n-gramlar, kesish (clipping), qisqalik jarimasi, geometrik oʻrtacha; bir xil silliqlash sozlamalari bilan `nltk.translate.bleu_score` orqali tekshirish; gap BLEU i va korpus BLEU i;
2. chrF — belgi n-gramlari boʻyicha F-oʻlchov — va nima uchun u oʻzbekcha qoʻshimchalardagi xatolarga yumshoqroq;
3. LCS uchun dinamik dasturlash jadvali orqali ROUGE-1, ROUGE-2 va ROUGE-L;
4. nomlangan obyektlar uchun aniqlik, toʻliqlik va F1 (obyektlar va tokenlar boʻyicha), `sklearn.metrics` bilan tekshirilgan micro va macro oʻrtachalar.

*«NLP algoritmlari» kursining 18-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. BLEU qadamma-qadam

BLEU tizim tarjimasini bitta yoki bir nechta etalon tarjima bilan mos keladigan n-gramlar boʻyicha solishtiradi, n = 1..4.

- Modifikatsiyalangan aniqlik: $p_n = \dfrac{\sum_g \min\big(\text{count}_{hyp}(g),\ \max_{ref} \text{count}_{ref}(g)\big)}{\sum_g \text{count}_{hyp}(g)}$, bu yerda $g$ — tarjimadagi n-gramlar. Kesish (clipping, $\min$) soʻzni takrorlab aniqlikni sunʼiy oshirishga yoʻl qoʻymaydi.
- Qisqalik jarimasi: agar $c > r$ boʻlsa, $BP = 1$, aks holda $BP = e^{\,1 - r/c}$; $c$ — tarjima uzunligi, $r$ — uzunligi tarjimaga eng yaqin etalon uzunligi.
- $\text{BLEU} = BP \cdot \exp\big(\tfrac14 \sum_{n=1}^{4} \log p_n\big)$, yaʼni jarima toʻrt aniqlikning geometrik oʻrtachasiga koʻpaytiriladi.

Misollar laboratoriyadan olingan. Bitta soʻz sinonimga almashtirilgan tarjimaning hisobini kuzatamiz: bu almashtirish 1 ta unigram, 2 ta bigram, 3 ta trigram va 4 ta toʻrtgramni buzadi.

Katakning ikkinchi yarmida kesish nima uchun kerakligi koʻrinadi. Toʻqqizta «biz» dan iborat tarjima kesishsiz $p_1 = 1$ oladi: har bir «biz» etalonda bor. Kesish bilan «biz» etalonda necha marta uchrasa, shuncha marta, yaʼni bir marta hisobga olinadi.

In [ ]:
import re, math, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ref1 = "Kecha biz doʻstlarimiz bilan Samarqandning eski koʻchalarida sayr qildik."
ref2 = "Biz kecha Samarqandning eski koʻchalarida doʻstlar bilan aylandik."
cands = {
    "Nusxa": "Kecha biz doʻstlarimiz bilan Samarqandning eski koʻchalarida sayr qildik.",
    "Sinonim": "Kecha biz doʻstlarimiz bilan Samarqandning qadimiy koʻchalarida sayr qildik.",
    "Soʻz tartibi": "Biz kecha Samarqandning eski koʻchalarida doʻstlarimiz bilan sayr qildik.",
    "Qoʻshimcha": "Kecha biz doʻstlarim bilan Samarqandning eski koʻchalarida sayr qildik.",
    "Qisqa": "Doʻstlarimiz bilan Samarqandning eski koʻchalarida sayr qildik.",
    "Takrorlar": "biz biz biz biz biz biz biz biz biz",
}

def tokenize(s):
    # kichik harflar, tinish belgilarisiz; ʻ va ʼ soʻz ichida qoladi
    return re.findall(r"\w+(?:[-'’]\w+)*", s.lower())

def ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

def clipped_counts(hyp, refs, n, clip=True):
    hyp_counts = Counter(ngrams(hyp, n))
    max_ref = Counter()                       # n-gram eng «saxiy» etalonda necha marta uchraydi
    for r in refs:
        for g, c in Counter(ngrams(r, n)).items():
            max_ref[g] = max(max_ref[g], c)
    credit = {g: (min(c, max_ref[g]) if clip else (c if max_ref[g] else 0)) for g, c in hyp_counts.items()}
    return hyp_counts, max_ref, credit

def closest_ref_len(refs, c):
    return min((abs(len(r) - c), len(r)) for r in refs)[1]   # teng boʻlsa — qisqarogʻi

def brevity_penalty(c, r):
    if c > r:
        return 1.0
    return 0.0 if c == 0 else math.exp(1 - r / c)

def trace(hyp_text, ref_texts, clip=True, max_n=4):
    hyp, refs = tokenize(hyp_text), [tokenize(r) for r in ref_texts]
    ps = []
    for n in range(1, max_n + 1):
        hyp_counts, max_ref, credit = clipped_counts(hyp, refs, n, clip)
        for g, c in hyp_counts.items():
            mark = "" if credit[g] else "   ✗"
            print(f"   {' '.join(g):<36} tarjimada {c}, etalonda {max_ref[g]} -> hisobga olindi {credit[g]}{mark}")
        matched, total = sum(credit.values()), max(sum(hyp_counts.values()), 1)
        ps.append(matched / total)
        print(f"p{n} = {matched}/{total} = {matched / total:.3f}\n")
    c, r = len(hyp), closest_ref_len(refs, len(hyp))
    bp = brevity_penalty(c, r)
    geo = math.exp(sum(math.log(p) for p in ps) / max_n) if min(ps) > 0 else 0.0
    name = "BLEU" if max_n == 4 else f"BLEU (n ≤ {max_n})"
    print(f"c = {c}, r = {r}, BP = {bp:.3f};  geometrik oʻrtacha = {geo:.3f};  {name} = {bp * geo:.3f}")

print(f"=== «{cands['Sinonim']}»")
trace(cands["Sinonim"], [ref1])

print()

for clip, name in [(False, "kesishsiz"), (True, "kesish bilan")]:
    print(f"=== {name}: «{cands['Takrorlar']}»")
    trace(cands["Takrorlar"], [ref1], clip=clip, max_n=1)
    print()

Endi `bleu` funksiyasi toʻliq va barcha misollar uchun `nltk.translate.bleu_score.sentence_bleu` bilan tekshirish. Silliqlashsiz bitta nol $p_n$ butun BLEU ni nolga aylantiradi (noldan logarifm). Silliqlashni bizda va nltk da bir xil sozlamalar bilan solishtiramiz:

- `SmoothingFunction().method1`: n-tartibli mosliklar boʻlmasa, 0 oʻrniga $\varepsilon / \text{maxraj}$ olinadi, $\varepsilon = 0{,}1$;
- `SmoothingFunction().method2`: $n \ge 2$ da $p_n$ ning surati va maxrajiga 1 qoʻshiladi (laboratoriyadagi «Silliqlash» belgisi).

Jadval ostidagi qatorda — barcha misollar va sozlamalar boʻyicha funksiyamizning nltk dan eng katta farqi ($10^{-16}$ tartibida yoki undan kichik, yaʼni yaxlitlash xatolarigacha mos).

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction

def bleu(hyp, refs, smoothing=None, eps=0.1, max_n=4):
    """hyp — tokenlar roʻyxati, refs — shunday roʻyxatlar roʻyxati; smoothing: None, "eps" (nltk method1), "add1" (nltk method2)"""
    matched, total = [], []
    for n in range(1, max_n + 1):
        hyp_counts, _, credit = clipped_counts(hyp, refs, n)
        matched.append(sum(credit.values()))
        total.append(max(sum(hyp_counts.values()), 1))   # nltk dagidek: maxraj 1 dan kichik emas
    if matched[0] == 0:
        return 0.0
    ps = []
    for n, (m, t) in enumerate(zip(matched, total), start=1):
        if smoothing == "add1" and n > 1:
            ps.append((m + 1) / (t + 1))
        elif smoothing == "eps" and m == 0:
            ps.append(eps / t)
        else:
            ps.append(m / t)
    if min(ps) == 0:
        return 0.0
    bp = brevity_penalty(len(hyp), closest_ref_len(refs, len(hyp)))
    return bp * math.exp(sum(math.log(p) for p in ps) / max_n)

NLTK_SMOOTHING = {None: SmoothingFunction().method0, "eps": SmoothingFunction().method1, "add1": SmoothingFunction().method2}
R1, R2 = tokenize(ref1), tokenize(ref2)
rows, max_diff = [], 0.0
for name, text in cands.items():
    h = tokenize(text)
    row = {"misol": name}
    for n in range(1, 5):
        _, _, cr = clipped_counts(h, [R1], n)
        row[f"p{n}"] = sum(cr.values()) / max(len(ngrams(h, n)), 1)
    row["BP"] = brevity_penalty(len(h), closest_ref_len([R1], len(h)))
    for col, refs, sm in [("BLEU", [R1], None), ("2 etalon", [R1, R2], None),
                          ("method1", [R1], "eps"), ("method2", [R1], "add1"), ("2 + method2", [R1, R2], "add1")]:
        ours = bleu(h, refs, sm)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")      # biror p_n = 0 boʻlsa, nltk ogohlantiradi
            ref_value = sentence_bleu(refs, h, smoothing_function=NLTK_SMOOTHING[sm])
        max_diff = max(max_diff, abs(ours - ref_value))
        row[col] = ours
    rows.append(row)
print(pd.DataFrame(rows).set_index("misol").round(3).to_string())
print(f"\nnltk bilan eng katta farq: {max_diff:.1e}")

### Gap BLEU i va korpus BLEU i

BLEU butun test korpusi uchun oʻylab topilgan. Korpus BLEU i avval barcha gaplar boʻyicha $p_n$ larning surat va maxrajlarini, shuningdek $c$ va $r$ uzunliklarini qoʻshadi, faqat keyin boʻladi (micro-oʻrtacha). Alohida gaplar BLEU ining oʻrtachasi — boshqa son: silliqlashsiz qisqa gapda bitta ham umumiy toʻrtgram boʻlmasa, u osongina 0 oladi. Kichik «test toʻplami»ning oltita gapida solishtiramiz.

In [ ]:
test_refs = [
    "Toshkentdan poyezd ertalab Samarqandga yetib keladi.",
    "Kutubxona yakshanbadan boshqa har kuni ochiq.",
    "Talabalar matematikadan imtihon topshirdi.",
    "Ertaga kunduzi havo quyoshli va iliq boʻladi, yogʻingarchilik kutilmaydi.",
    "Akam shahar kasalxonasida shifokor boʻlib ishlaydi.",
    "Biz bozordan yangi mevalar sotib oldik.",
]
test_hyps = [
    "Toshkentdan poyezd ertalab Samarqandga keladi.",
    "Kutubxona yakshanbadan tashqari har kuni ishlaydi.",
    "Talabalar matematikadan imtihon topshirdi.",
    "Ertaga havo quyoshli boʻladi.",
    "Akam shahar shifoxonasida vrach boʻlib ishlaydi.",
    "Biz yangi mevalarni bozordan sotib oldik.",
]

hyps = [tokenize(h) for h in test_hyps]
refs_list = [[tokenize(r)] for r in test_refs]

def corpus_bleu_ours(hyps, refs_list, max_n=4):
    matched, total = np.zeros(max_n), np.zeros(max_n)
    c = r = 0
    for hyp, refs in zip(hyps, refs_list):
        for n in range(1, max_n + 1):
            hyp_counts, _, credit = clipped_counts(hyp, refs, n)
            matched[n - 1] += sum(credit.values())     # avval barcha gaplar boʻyicha qoʻshamiz...
            total[n - 1] += sum(hyp_counts.values())
        c += len(hyp)
        r += closest_ref_len(refs, len(hyp))
    ps = matched / total                               # ...keyin boʻlamiz
    print("p_n =", np.round(ps, 3), f" c = {c}, r = {r}")
    return brevity_penalty(c, r) * math.exp(np.mean(np.log(ps))) if ps.min() > 0 else 0.0

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sent_plain = [sentence_bleu(refs, h) for h, refs in zip(hyps, refs_list)]
    sent_smooth = [sentence_bleu(refs, h, smoothing_function=SmoothingFunction().method2) for h, refs in zip(hyps, refs_list)]
for h, b0, b2 in zip(test_hyps, sent_plain, sent_smooth):
    print(f"{h:<52} gap BLEU = {b0:.3f}   method2 = {b2:.3f}")
print(f"\ngaplar BLEU ining oʻrtachasi: {np.mean(sent_plain):.3f} (method2: {np.mean(sent_smooth):.3f})")
ours = corpus_bleu_ours(hyps, refs_list)
print(f"korpus BLEU: bizniki {ours:.4f}, nltk.corpus_bleu {corpus_bleu(refs_list, hyps):.4f}")

Korpus BLEU i gaplar BLEU ining oʻrtachasiga teng emas: bitta ham umumiy toʻrtgrami yoʻq, lekin ancha durust tarjimalar BLEU = 0 oldi, korpusda esa ularning mosliklari baribir hisobga olinadi. Shuning uchun hisobotlarda korpus BLEU i keltiriladi, alohida gap BLEU i esa faqat silliqlash bilan hisoblanadi.

### chrF: belgi n-gramlari boʻyicha F-oʻlchov

BLEU uchun «doʻstlarimiz» va «doʻstlarim» xuddi «doʻstlarimiz» va «stol» kabi turlicha. chrF (Popović, 2015) boʻshliqlarni olib tashlaydi va belgi n-gramlarini solishtiradi, n = 1..6. Har bir n uchun aniqlik $P_n$ va toʻliqlik $R_n$, keyin ularning oʻrtachalari $P$ va $R$ hisoblanadi, soʻng toʻliqlik $\beta = 2$ marta muhimroq boʻlgan F-oʻlchov: $\text{chrF} = \dfrac{(1+\beta^2)\,P R}{\beta^2 P + R}$.

nltk da `sentence_chrf` bor, lekin u $P$ va $R$ ni emas, alohida tartiblarning F-oʻlchovlari $F_n$ ni oʻrtachalaydi (standart holatda $\beta = 3$ oladi). Shuning uchun funksiyamiz ikkala usulni ham biladi: nltk usulini $\beta = 2$ da nltk bilan tekshiramiz, standart usulni (sacrebleu va laboratoriyadagidek) esa BLEU bilan solishtiramiz. Ular qisqa satrlarda sezilarli farq qiladi: nltk usuli satrda yoʻq $n$ tartib uchun $F_n = 0$ deb oladi, standart usul esa bunday tartibni tashlab ketadi. Bizning misollarda sonlar deyarli bir xil.

**Nima uchun chrF oʻzbek tiliga mos.** Agglyutinativ tilda bitta grammatik maʼno — bitta qoʻshimcha. Bitta maʼnodagi xato (doʻstlarimiz → doʻstlarim) asos mos kelgan boʻlsa ham, butun soʻzni BLEU uchun «begona» qiladi. chrF asosning umumiy belgi n-gramlarini hisobga oladi va faqat oʻzgargan qoʻshimcha uchun jarima beradi. Katakning ikkinchi qismida bitta qoʻshimchasida xato boʻlgan alohida soʻzlarni koʻramiz: BLEU uchun har biri shunchaki mos kelmagan (0), chrF esa qisman baho beradi.

In [ ]:
from nltk.translate.chrf_score import sentence_chrf

def char_ngrams(text, n):
    s = re.sub(r"\s+", "", text)                       # boʻshliqlar olib tashlanadi
    return Counter(s[i:i + n] for i in range(len(s) - n + 1))

def chrf(hyp, ref, n_max=6, beta=2.0, average="pr"):
    """average="pr": n boʻyicha P va R oʻrtachasi, keyin F (sacrebleu, laboratoriya); average="f": n boʻyicha F_n oʻrtachasi (nltk)"""
    b2 = beta ** 2
    Ps, Rs, Fs = [], [], []
    for n in range(1, n_max + 1):
        h, r = char_ngrams(hyp, n), char_ngrams(ref, n)
        if not h or not r:
            Fs.append(0.0)
            continue
        m = sum((h & r).values())                      # Counter dagi & — elementlar boʻyicha minimum
        p, rc = m / sum(h.values()), m / sum(r.values())
        Ps.append(p); Rs.append(rc)
        Fs.append((1 + b2) * p * rc / (b2 * p + rc) if p + rc else 0.0)
    if average == "f":
        return float(np.mean(Fs))
    P, R = (np.mean(Ps), np.mean(Rs)) if Ps else (0.0, 0.0)
    return float((1 + b2) * P * R / (b2 * P + R)) if P + R else 0.0

ref_text = " ".join(tokenize(ref1))
rows = []
for name, text in cands.items():
    hyp_text = " ".join(tokenize(text))
    rows.append({"misol": name,
                 "nltk (n boʻyicha F oʻrtachasi)": sentence_chrf(ref_text, hyp_text, beta=2.0),
                 "bizniki, nltk usuli": chrf(hyp_text, ref_text, average="f"),
                 "bizniki, standart": chrf(hyp_text, ref_text),
                 "BLEU": bleu(tokenize(text), [tokenize(ref1)])})
table = pd.DataFrame(rows).set_index("misol")
print(table.round(3).to_string())
print(f"\nnltk bilan eng katta farq: {(table['nltk (n boʻyicha F oʻrtachasi)'] - table['bizniki, nltk usuli']).abs().max():.1e}\n")


import os

word_pairs = [   # (etalon soʻzi, tarjima soʻzi): bitta qoʻshimchada xato
    ("doʻstlarimiz", "doʻstlarim"),
    ("kitoblarimizdan", "kitoblarimizda"),
    ("oʻqituvchimiz", "oʻqituvchimizga"),
    ("talabalarimizga", "talabalarimizni"),
    ("kutubxonaga", "kutubxonada"),
    ("samarqandning", "samarqand"),
    ("yozdik", "yozdim"),
    ("uyga", "uyda"),
]

for ref_word, hyp_word in word_pairs:
    shared = len(os.path.commonprefix([ref_word, hyp_word]))     # umumiy boshlangʻich qism uzunligi
    print(f"{ref_word:<16} -> {hyp_word:<16} BLEU: 0   umumiy harflar: {shared:2d} / {len(ref_word):2d}   chrF = {chrf(hyp_word, ref_word):.2f}")

ax = table[["BLEU", "bizniki, standart"]].rename(columns={"bizniki, standart": "chrF"}).plot.bar(figsize=(8, 3.5), color=["#2B55C9", "#17907E"], rot=15)
ax.set_ylabel("baho"); ax.set_ylim(0, 1.05); ax.set_title("Laboratoriya misollari uchun BLEU va chrF (bitta etalon)")
plt.tight_layout()
plt.show()

chrF bahosi umumiy qism uzunligi bilan oʻsadi: 14 ta umumiy harfda (kitoblarimizdan → kitoblarimizda) u 0,9 dan yuqori. Oʻzbek soʻz shakllari uzun, qoʻshimchalar esa oʻzgarmaydigan asos oxiriga qoʻshiladi, shuning uchun bitta qoʻshimchadagi xato uzun umumiy qismni saqlab qoladi. Qisqa soʻzlar (uyga → uyda) har qanday tilda kam baho oladi. Teskari tomoni: agar maʼno xatosi qisqa qoʻshimcha bilan ifodalangan boʻlsa, chrF haqiqiy xatoni ham kechirib yuborishi mumkin: «kutubxonaga» va «kutubxonada» — butunlay boshqa joy, baho esa 0,77.

### Standart vositalarda (faqat Colab)

Mashina tarjimasiga oid maqolalarda BLEU va chrF `sacrebleu` paketi bilan hisoblanadi. Colab da u oʻrnatilmagan: katak uni oʻrnatadi va funksiyalarimiz bilan solishtiradi. Standart holatda sacrebleu matnni oʻzi tokenlaydi va registrni hisobga oladi, shuning uchun bir xil sonlar olish uchun unga kichik harfdagi tokenlarimizni va `tokenize="none"` ni beramiz.

In [ ]:
# Faqat Colab uchun: mashina tarjimasi maqolalarida BLEU va chrF uchun standart paket sacrebleu ni oʻrnatamiz
!pip install -q sacrebleu
try:
    import sacrebleu
    hyp_strings = [" ".join(h) for h in hyps]
    ref_strings = [" ".join(rs[0]) for rs in refs_list]
    # matnlar allaqachon tokenlangan va kichik harflarda, shuning uchun tokenize="none" va silliqlashsiz
    sb = sacrebleu.corpus_bleu(hyp_strings, [ref_strings], tokenize="none", smooth_method="none")
    print(f"BLEU: sacrebleu {sb.score / 100:.4f}   bizniki {corpus_bleu_ours(hyps, refs_list):.4f}")
    # standart holatda sacrebleu oʻzi tokenlaydi (13a) va registrni hisobga oladi: natija boshqacha boʻladi
    print("sacrebleu, xom matnlar:", sacrebleu.corpus_bleu(test_hyps, [test_refs]))
    chrf_metric = sacrebleu.CHRF()           # char_order=6, word_order=0, beta=2
    for h, r in zip(hyp_strings, ref_strings):
        print(f"chrF: sacrebleu {chrf_metric.sentence_score(h, [r]).score / 100:.4f}   bizniki {chrf(h, r):.4f}   {h}")
except Exception as e:
    print("Ishlamadi (internet va paket kerak: sacrebleu):", e)

## 2. ROUGE-1, ROUGE-2 va ROUGE-L

ROUGE avtomatik referatlarni baholaydi (13-mavzu). BLEU kabi u ham umumiy n-gramlarni sanaydi, lekin undagi asosiy kattalik — toʻliqlik: referat etalonning qancha qismini qamrab olgan.

- ROUGE-N: $P = \dfrac{\text{umumiy n-gramlar}}{\text{referatdagi n-gramlar}}$, $R = \dfrac{\text{umumiy n-gramlar}}{\text{etalondagi n-gramlar}}$, $F_1 = \dfrac{2PR}{P+R}$.
- ROUGE-L: n-gramlar oʻrniga eng uzun umumiy qism ketma-ketlik (LCS) olinadi — ikkala matnda bir xil tartibda, lekin ketma-ket boʻlishi shart boʻlmagan soʻzlar. Uning uzunligini dinamik dasturlash topadi: agar $ref_i = cand_j$ boʻlsa, $T[i][j] = T[i-1][j-1] + 1$, aks holda $T[i][j] = \max(T[i-1][j],\ T[i][j-1])$. Shunda $P = LCS / |cand|$, $R = LCS / |ref|$.

Laboratoriyadagi «Oʻrin almashgan» misoli uchun jadvalni toʻldiramiz va pastki oʻng burchakdan teskari yurish bilan qism ketma-ketlikning oʻzini topamiz (rasmda ajratib koʻrsatilgan).

In [ ]:
rouge_ref = "Toshkent hokimligi shahar markazida yangi kutubxona qurishga qaror qildi"
summaries = {
    "Oʻrin almashgan": "Yangi kutubxona qurishga shahar markazida Toshkent hokimligi qaror qildi",
    "Qayta ifoda": "Toshkentda shahar markazida yangi kutubxona quriladi",
    "Teskari tartib": "qildi qaror qurishga kutubxona yangi markazida shahar hokimligi Toshkent",
    "Uzun": "Kecha Toshkent hokimligi nihoyat shahar markazida katta yangi kutubxona qurishga qaror qildi",
}

def prf(overlap, n_cand, n_ref):
    overlap = int(overlap)
    P = overlap / n_cand if n_cand else 0.0
    R = overlap / n_ref if n_ref else 0.0
    return P, R, (2 * P * R / (P + R) if P + R else 0.0)

def rouge_n(cand, ref, n):
    c, r = Counter(ngrams(cand, n)), Counter(ngrams(ref, n))
    return prf(sum((c & r).values()), sum(c.values()), sum(r.values()))

def lcs_table(ref, cand):
    T = np.zeros((len(ref) + 1, len(cand) + 1), dtype=int)
    for i in range(1, len(ref) + 1):
        for j in range(1, len(cand) + 1):
            if ref[i - 1] == cand[j - 1]:
                T[i, j] = T[i - 1, j - 1] + 1                 # soʻzlar mos: diagonal + 1
            else:
                T[i, j] = max(T[i - 1, j], T[i, j - 1])       # aks holda «yuqori» va «chap»dan kattasi
    return T

def lcs_backtrack(T, ref, cand):
    i, j, words, path = len(ref), len(cand), [], []
    while i > 0 and j > 0:
        if ref[i - 1] == cand[j - 1]:
            words.append(ref[i - 1]); path.append((i, j)); i -= 1; j -= 1
        elif T[i - 1, j] >= T[i, j - 1]:
            i -= 1
        else:
            j -= 1
    return words[::-1], path[::-1]

def rouge_l(cand, ref):
    T = lcs_table(ref, cand)
    return prf(T[-1, -1], len(cand), len(ref))

ref = tokenize(rouge_ref)
cand = tokenize(summaries["Oʻrin almashgan"])
T = lcs_table(ref, cand)
print(pd.DataFrame(T, index=["∅"] + ref, columns=["∅"] + cand).to_string())
words, path = lcs_backtrack(T, ref, cand)
print(f"\nLCS = «{' '.join(words)}», uzunligi {len(words)}")
P, R, F = rouge_l(cand, ref)
print(f"ROUGE-L: P = {len(words)}/{len(cand)} = {P:.3f}, R = {len(words)}/{len(ref)} = {R:.3f}, F1 = {F:.3f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(T, cmap="Blues", vmin=0, vmax=2 * T.max())
for i in range(T.shape[0]):
    for j in range(T.shape[1]):
        on_path = (i, j) in path
        ax.text(j, i, T[i, j], ha="center", va="center", fontsize=10,
                fontweight="bold" if on_path else "normal", color="#C94B2B" if on_path else "black")
        if on_path:
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="#C94B2B", linewidth=2))
ax.set_xticks(range(len(cand) + 1)); ax.set_xticklabels(["∅"] + cand, rotation=40, ha="right")
ax.set_yticks(range(len(ref) + 1)); ax.set_yticklabels(["∅"] + ref)
ax.set_title("LCS jadvali: qatorlar — etalon, ustunlar — referat")
plt.tight_layout()
plt.show()

Endi laboratoriyaning barcha misollari. nltk da tayyor ROUGE funksiyasi yoʻq, shuning uchun LCS ni mustaqil usulda tekshiramiz. Agar faqat qoʻshish va oʻchirishga ruxsat berilsa (almashtirish 2 turadi, xuddi oʻchirish va qoʻshish kabi), tahrir masofasi $|a| + |b| - 2\cdot LCS$ ga teng. Demak, `nltk.edit_distance(..., substitution_cost=2)` LCS ni boshqa algoritm bilan beradi (11-mavzu).

In [ ]:
from nltk import edit_distance

rows = []
for name, text in summaries.items():
    cand = tokenize(text)
    p1, r1, f1 = rouge_n(cand, ref, 1)
    _, _, f2 = rouge_n(cand, ref, 2)
    pl, rl, fl = rouge_l(cand, ref)
    lcs = lcs_table(ref, cand)[-1, -1]
    d = edit_distance(ref, cand, substitution_cost=2)   # faqat qoʻshish va oʻchirish
    rows.append({"misol": name, "R1 P": p1, "R1 R": r1, "R1 F": f1, "R2 F": f2, "RL P": pl, "RL R": rl, "RL F": fl,
                 "LCS": lcs, "nltk.edit_distance boʻyicha LCS": (len(ref) + len(cand) - d) // 2})
print(pd.DataFrame(rows).set_index("misol").round(3).to_string())

«Teskari tartib» maʼnosiz matn boʻlsa ham ROUGE-1 = 1 oladi, ROUGE-L esa uni fosh qiladi. «Uzun» referatda butun etalon bor: toʻliqlik 1, aniqlik pastroq. «Qayta ifoda» soʻz shakllari tufayli yutqazadi: «Toshkentda» va «Toshkent» turli tokenlar. Oʻzbek tilida bu ayniqsa koʻp uchraydi, shuning uchun ROUGE ni lemmalar yoki asoslar boʻyicha hisoblash foydali (3-topshiriq).

### Standart vositalarda (faqat Colab)

Standart `rouge-score` (Google) paketi standart holatda matnda faqat lotin harflari va raqamlarni qoldiradi. Oʻzbekcha oʻqidi u uchun o va qidi ga boʻlinib ketadi, kirill yozuvidagi matn esa butunlay yoʻqoladi. Katak buni koʻrsatadi va oʻz tokenizatorimizni berib tuzatadi.

In [ ]:
# Faqat Colab uchun: rouge-score paketi (Google)
!pip install -q rouge-score
try:
    from rouge_score import rouge_scorer
    text = summaries["Oʻrin almashgan"]
    default = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"])
    from rouge_score import tokenizers
    print("standart tokenizator:", tokenizers.DefaultTokenizer().tokenize(rouge_ref))
    print("standart tokenizator:", tokenizers.DefaultTokenizer().tokenize("Oʻqituvchimiz kitob oʻqidi"))
    print("   ", {k: round(v.fmeasure, 3) for k, v in default.score(rouge_ref, text).items()})

    class OurTokenizer:                      # rouge-score tokenize metodi bor har qanday obyektni qabul qiladi
        def tokenize(self, s):
            return tokenize(s)

    fixed = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=OurTokenizer())
    for name, text in summaries.items():
        scores = fixed.score(rouge_ref, text)    # argumentlar tartibi: (etalon, referat)
        cand = tokenize(text)
        ours = (rouge_n(cand, ref, 1)[2], rouge_n(cand, ref, 2)[2], rouge_l(cand, ref)[2])
        print(f"{name:<18} rouge-score: {[round(scores[k].fmeasure, 3) for k in ['rouge1', 'rouge2', 'rougeL']]}   bizniki: {[round(x, 3) for x in ours]}")
except Exception as e:
    print("Ishlamadi (internet va paket kerak: rouge-score):", e)

## 3. Nomlangan obyektlar uchun aniqlik, toʻliqlik va F1

NER da (9-mavzu) obyektlar toʻplamlari solishtiriladi: etalon (gold) va model javobi (predicted). TP — toʻgʻri topilganlar, FP — ortiqchalari, FN — topilmaganlari:
$P = \dfrac{TP}{TP+FP}$, $R = \dfrac{TP}{TP+FN}$, $F_1 = \dfrac{2PR}{P+R}$.

«Toʻgʻri» nimani bildiradi? Aniq solishtirishda (CoNLL dagidek) chegaralar ham, tur ham mos kelishi kerak. Qisman solishtirishda bir xil turdagi etalon obyekt bilan kesishma yetarli. Laboratoriyadagi gapni olamiz: tashkilotni model toʻgʻri topdi, shaxsda chegarada xato qildi, joyni tashkilot deb atadi va ortiqcha obyekt qoʻshdi. Eʼtibor bering: aniq solishtirishda chegara xatosi ikki marta hisoblanadi — ham FP, ham FN.

In [ ]:
tokens = ["Kecha", "Oʻzbekiston", "Milliy", "universitetidan", "Dilnoza", "Rustamovna", "Karimova",
          "Samarqand", "shahridagi", "forumda", "soʻzga", "chiqdi"]
# obyekt = (gap raqami, birinchi token, oxirgi token, tur)
gold = [(0, 1, 3, "ORG"), (0, 4, 6, "PER"), (0, 7, 7, "LOC")]
pred = [(0, 1, 3, "ORG"),    # toʻgʻri
        (0, 4, 5, "PER"),    # chegara: «Karimova» tushib qolgan
        (0, 7, 7, "ORG"),    # tur: joy tashkilot deb topilgan
        (0, 9, 9, "ORG")]    # ortiqcha: «forumda»

def show_entity(e, toks):
    return f"[{' '.join(toks[e[1]:e[2] + 1])}]{e[3]}"

def matches(p, g, mode):
    if mode == "exact":                                    # chegaralar va tur mos
        return p == g
    same_sent_type = p[0] == g[0] and p[3] == g[3]         # partial: tur bir xil va kesishma bor
    return same_sent_type and p[1] <= g[2] and g[1] <= p[2]

def ner_prf(gold, pred, mode="exact", verbose=False, toks=None):
    used, tp = set(), 0
    for p in pred:
        hit = next((g for g in gold if g not in used and matches(p, g, mode)), None)
        if hit is not None:
            used.add(hit); tp += 1
        if verbose:
            print(f"   {show_entity(p, toks):<34} -> {'TP' if hit else 'FP'}")
    if verbose:
        for g in gold:
            if g not in used:
                print(f"   {show_entity(g, toks):<34} -> FN (topilmadi)")
    fp, fn = len(pred) - tp, len(gold) - tp
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    return tp, fp, fn, P, R, F

for mode, name in [("exact", "aniq"), ("partial", "qisman")]:
    print(f"=== {name}")
    tp, fp, fn, P, R, F = ner_prf(gold, pred, mode, verbose=True, toks=tokens)
    print(f"   TP = {tp}, FP = {fp}, FN = {fn};  P = {P:.3f}, R = {R:.3f}, F1 = {F:.3f}\n")

### Obyektlar darajasi va tokenlar darajasi

BIO belgilashdagi olti gapli korpusda ikki darajani solishtiramiz: obyektlar boʻyicha (aniq moslik) va tokenlar boʻyicha (har bir tokenda uning turi belgisi, O baholanmaydi). Tokenlar darajasida chegara xatosi bitta token turadi, obyektlar darajasida esa butun obyekt, shuning uchun tokenlar boʻyicha F1 deyarli har doim yuqoriroq. Turlar boʻyicha jadvalda model qayerda adashishi koʻrinadi. Oʻzbekcha belgilashga eʼtibor bering: Toshkentdan, Samarqandga kabi soʻzlar kelishik qoʻshimchasi bilan birga butunligicha LOC deb belgilanadi.

`sklearn.metrics.precision_recall_fscore_support` bilan tekshiramiz. Obyektlar darajasi uchun obyektlar toʻplamlarining birlashmasi boʻyicha (etalon, bashorat) juftliklarini tuzamiz: juftsiz topilgan obyektga etalonda «O» toʻgʻri keladi (FP), topilmaganiga esa bashoratda «O» (FN).

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

ner_corpus = [   # (gap, etalon BIO belgilari, model belgilari)
    ("Dilnoza Karimova Toshkentdan Samarqandga koʻchib keldi",
     "B-PER I-PER B-LOC B-LOC O O", "B-PER I-PER B-LOC B-LOC O O"),
    ("Oʻzbekiston Milliy universiteti talabalari Buxoroga borishdi",
     "B-ORG I-ORG I-ORG O B-LOC O", "B-LOC B-ORG I-ORG O B-LOC O"),
    ("Professor Akmal Rahimov Toshkent davlat texnika universitetida ishlaydi",
     "O B-PER I-PER B-ORG I-ORG I-ORG I-ORG O", "B-PER I-PER I-PER B-ORG I-ORG I-ORG I-ORG O"),
    ("Kecha Xiva shahrida festival boʻldi",
     "O B-LOC O O O", "O B-ORG O O O"),
    ("Samarqand davlat universiteti talabalari Navoiyga borishdi",
     "B-ORG I-ORG I-ORG O B-LOC O", "B-LOC O O O B-LOC O"),
    ("Bobur Sherzodga xat yozdi",
     "B-PER B-PER O O", "B-PER I-PER O O"),
]

def bio_entities(tags, sent_id):
    ents, start, typ = [], None, None
    for i, t in enumerate(tags + ["O"]):
        # obyekt O, B- yoki boshqa turdagi I- da tugaydi
        if t == "O" or t.startswith("B-") or t[2:] != typ:
            if start is not None:
                ents.append((sent_id, start, i - 1, typ))
            start, typ = (i, t[2:]) if t != "O" else (None, None)
    return ents

gold_ents = [e for k, (s, g, p) in enumerate(ner_corpus) for e in bio_entities(g.split(), k)]
pred_ents = [e for k, (s, g, p) in enumerate(ner_corpus) for e in bio_entities(p.split(), k)]
types = ["LOC", "ORG", "PER"]

def per_type_table(counts):
    rows = {}
    for t, (tp, fp, fn) in counts.items():
        P = tp / (tp + fp) if tp + fp else 0.0
        R = tp / (tp + fn) if tp + fn else 0.0
        rows[t] = [tp, fp, fn, P, R, 2 * P * R / (P + R) if P + R else 0.0]
    df = pd.DataFrame(rows, index=["TP", "FP", "FN", "P", "R", "F1"]).T
    TP, FP, FN = df["TP"].sum(), df["FP"].sum(), df["FN"].sum()
    micro_P, micro_R = TP / (TP + FP), TP / (TP + FN)
    df.loc["micro"] = [TP, FP, FN, micro_P, micro_R, 2 * micro_P * micro_R / (micro_P + micro_R)]
    df.loc["macro"] = [np.nan, np.nan, np.nan] + list(df.loc[types, ["P", "R", "F1"]].mean())
    return df

# obyektlar darajasi: aniq moslik
ent_counts = {t: (len([e for e in pred_ents if e[3] == t and e in gold_ents]),
                  len([e for e in pred_ents if e[3] == t and e not in gold_ents]),
                  len([e for e in gold_ents if e[3] == t and e not in pred_ents])) for t in types}
# tokenlar darajasi: B-/I- prefikssiz belgi
y_true = [t.split("-")[-1] for s, g, p in ner_corpus for t in g.split()]
y_pred = [t.split("-")[-1] for s, g, p in ner_corpus for t in p.split()]
tok_counts = {t: (sum(a == t and b == t for a, b in zip(y_true, y_pred)),
                  sum(a != t and b == t for a, b in zip(y_true, y_pred)),
                  sum(a == t and b != t for a, b in zip(y_true, y_pred))) for t in types}

ent_table, tok_table = per_type_table(ent_counts), per_type_table(tok_counts)
print("=== obyektlar boʻyicha\n" + ent_table.round(3).to_string(na_rep=""))
print("\n=== tokenlar boʻyicha\n" + tok_table.round(3).to_string(na_rep=""))

# sklearn orqali tekshirish; obyektlar uchun (etalon, bashorat) juftliklari, mos kelmagan obyektga "O" toʻgʻri keladi
union = sorted(set(gold_ents) | set(pred_ents))
ent_true = [e[3] if e in gold_ents else "O" for e in union]
ent_pred = [e[3] if e in pred_ents else "O" for e in union]
print()
for level, (yt, yp), table in [("obyektlar boʻyicha", (ent_true, ent_pred), ent_table), ("tokenlar boʻyicha", (y_true, y_pred), tok_table)]:
    for avg in ["micro", "macro"]:
        P, R, F, _ = precision_recall_fscore_support(yt, yp, labels=types, average=avg, zero_division=0)
        ok = np.allclose([P, R, F], table.loc[avg, ["P", "R", "F1"]].astype(float))
        print(f"sklearn {level:<18} {avg}: P = {P:.3f}, R = {R:.3f}, F1 = {F:.3f}   {'mos ✓' if ok else '✗'}")

### Micro va macro

Sinflar bir nechta boʻlsa, F1 ikki usulda oʻrtachalanadi:

- **macro-F1** — sinflar F1 ining oddiy oʻrtachasi, kam uchraydigan sinf koʻp uchraydigan sinf bilan bir xil vaznga ega (sklearn da bu oʻrtacha P va R dan olingan F1 emas, F1 larning oʻrtachasi);
- **micro-F1** — avval barcha sinflarning TP, FP va FN lari qoʻshiladi, keyin bitta F1 hisoblanadi, koʻp uchraydigan sinflar koʻproq vaznga ega. Agar har bir misolda aynan bitta javob boʻlsa, micro-F1 accuracy ga teng: har bir xato bir vaqtda bir sinfning FP si va boshqa sinfning FN i.

Xatoliklar matritsalari laboratoriyadan olingan (yangiliklarni mavzular boʻyicha tasniflagich): qatorlar — toʻgʻri sinf, ustunlar — model javobi.

In [ ]:
from sklearn.metrics import f1_score, classification_report

classes = ["sport", "iqtisodiyot", "madaniyat"]
presets = {
    "Muvozanatli sinflar": [[18, 1, 1], [2, 16, 2], [1, 2, 17]],
    "Kam uchraydigan sinf": [[50, 2, 1], [3, 30, 1], [3, 2, 1]],
    "Doim «sport»": [[55, 0, 0], [35, 0, 0], [5, 0, 0]],
}

def f1_from_matrix(M):
    M = np.array(M)
    tp = np.diag(M)
    fp = M.sum(axis=0) - tp          # ustunning qolgan kataklari
    fn = M.sum(axis=1) - tp          # qatorning qolgan kataklari
    P = np.divide(tp, tp + fp, out=np.zeros(len(tp)), where=(tp + fp) > 0)   # 0/0 ni 0 deb hisoblaymiz
    R = np.divide(tp, tp + fn, out=np.zeros(len(tp)), where=(tp + fn) > 0)
    F = np.divide(2 * P * R, P + R, out=np.zeros(len(tp)), where=(P + R) > 0)
    micro_P = tp.sum() / (tp.sum() + fp.sum())
    micro_R = tp.sum() / (tp.sum() + fn.sum())
    return P, R, F, F.mean(), 2 * micro_P * micro_R / (micro_P + micro_R), tp.sum() / M.sum()

def expand(M):
    # xatoliklar matritsasidan sklearn uchun javoblar roʻyxatiga
    y_true, y_pred = [], []
    for i, row in enumerate(M):
        for j, n in enumerate(row):
            y_true += [classes[i]] * n
            y_pred += [classes[j]] * n
    return y_true, y_pred

summary = {}
for name, M in presets.items():
    P, R, F, macro, micro, acc = f1_from_matrix(M)
    yt, yp = expand(M)
    sk_macro = f1_score(yt, yp, labels=classes, average="macro", zero_division=0)
    sk_micro = f1_score(yt, yp, labels=classes, average="micro", zero_division=0)
    print(f"=== {name}")
    print(pd.DataFrame({"P": P, "R": R, "F1": F}, index=classes).round(3).to_string())
    print(f"macro-F1 = {macro:.3f} (sklearn {sk_macro:.3f})   micro-F1 = {micro:.3f} (sklearn {sk_micro:.3f})   accuracy = {acc:.3f}\n")
    summary[name] = {"micro-F1": micro, "macro-F1": macro}

print(classification_report(*expand(presets[list(presets)[1]]), labels=classes, digits=3, zero_division=0))

ax = pd.DataFrame(summary).T.plot.bar(figsize=(7, 3.5), color=["#2B55C9", "#C94B2B"], rot=0)
ax.set_ylim(0, 1.05); ax.set_title("Uchta xatoliklar matritsasida micro-F1 va macro-F1")
plt.tight_layout()
plt.show()

## 4. Topshiriqlar

1. **Oʻz tarjimalaringiz.** `ref1` va `ref2` uchun ikkita tarjima oʻylab toping: (a) odam uchun tushunarli, lekin silliqlashsiz BLEU = 0; (b) BLEU yuqori, lekin maʼnosi teskari (masalan, inkor «-ma» qoʻshimchasi bilan: sayr qilmadik). Ular uchun BLEU (`method2` bilan) va chrF ni hisoblang. Qaysi metrika muammoni sezdi, qaysi biri yoʻq?
2. **chrF++.** chrF++ da belgi n-gramlariga (n = 1..6) soʻz unigramlari va bigramlari qoʻshiladi, $P$ va $R$ esa barcha 8 tartib boʻyicha oʻrtachalanadi. `chrf_pp` funksiyasini yozing va BLEU, chrF va chrF++ ni «Sinonim» va «Qoʻshimcha» misollarida solishtiring. Colab da `sacrebleu.CHRF(word_order=2)` bilan tekshiring.
3. **Asoslar boʻyicha ROUGE.** «Qayta ifoda» uchun har bir soʻzni dastlabki 5 harfgacha qisqartirib (1-amaliy mashgʻulotdagidek lemmatizatsiyaning qoʻpol oʻrinbosari), ROUGE-1, ROUGE-2 va ROUGE-L ni hisoblang. Baholar qanchaga oshdi? Qaysi soʻzlar endi mos keldi, qaysilari hali ham mos emas va nima uchun?
4. **Obyektlarni solishtirishning yana bir usuli.** `matches` funksiyasiga `"boundary"` rejimini qoʻshing: tur muhim emas, chegaralar aniq mos kelishi kerak. 3-boʻlimdagi korpusda (`gold_ents`, `pred_ents`) uchala rejimda P, R va F1 ni hisoblang. Modelning qaysi xatolari koʻproq: turlardami yoki chegaralardami?
5. **Micro va macro.** Micro-F1 ≥ 0,9, macro-F1 esa ≤ 0,6 boʻlgan `my_matrix` xatoliklar matritsasini (3×3, 3-boʻlimdagi sinflar) bering. Uni `f1_score` orqali tekshiring va `average="weighted"` ni ham hisoblang. Weighted ikki oʻrtachadan qaysi biriga yaqinroq va nima uchun? Soʻng oʻz-oʻzini tekshirish katagini ishga tushiring.

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# 5-topshiriq uchun oʻz-oʻzini tekshirish: yechgandan keyin ishga tushiring
M = globals().get("my_matrix")
if M is None:
    print("Avval 5-topshiriqda my_matrix ni bering.")
else:
    _, _, _, macro, micro, _ = f1_from_matrix(M)
    print(("✓" if micro >= 0.9 else "✗"), f"micro-F1 ≥ 0,9: micro-F1 = {micro:.3f}")
    print(("✓" if macro <= 0.6 else "✗"), f"macro-F1 ≤ 0,6: macro-F1 = {macro:.3f}")